# YOLO26s Half-Resolution Retraining — Colab (Phase 7)

Trains YOLO26s on GDINO pseudo-labeled **half-resolution** (1352×760) frames.

| Parameter | Previous (26l, full-res) | This run (26s, half-res) |
|-----------|--------------------------|---------------------------|
| model | yolo26l (26M params) | **yolo26s (10M params)** |
| freeze | 5 | **10** |
| mosaic | 0.5 | **0.0** |
| scale | 0.3 | **0.5** |
| translate | 0.1 | **0.15** |
| imgsz | 1280 | 1280 |
| batch | auto | auto (−1) |
| training frames | 521 (full-res) | ~2071 (half-res, 4×) |
| val frames | 522 (full-res) | ~2076 (half-res, 4×) |

**Drive directory structure** (`MyDrive/TreeLearn/tree_yolo26s_halfres/`):
```
TreeLearn/
  tree_yolo26s_halfres/
    images/train/*.jpg   (~2071 files)
    images/val/*.jpg     (~2076 files)
    labels/train/*.txt   (~2071 files)
    labels/val/*.txt     (~2076 files)
    weights/             (written after training)
```

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.56

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/TreeLearn/tree_yolo26s_halfres')
DATA_DIR  = Path('/content/data/pseudo_labels_half/all_4videos')
RUN_DIR   = Path('/content/runs/detect/tree_yolo26s_halfres')

assert Path('/content/drive/MyDrive').exists(), 'Drive not mounted — run drive.mount() first'

# Verify Drive dataset structure
for p in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    d = DRIVE_DIR / p
    n = len(list(d.glob('*'))) if d.exists() else -1
    print(f'{p}: {n} files  [{"OK" if n > 0 else "MISSING"}]')

In [ ]:
# Copy dataset from Drive to local SSD (faster I/O during training)
import shutil

print('Copying dataset to local SSD...')
for split in ('train', 'val'):
    for kind in ('images', 'labels'):
        src = DRIVE_DIR / kind / split
        dst = DATA_DIR  / kind / split
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n = len(list(dst.glob('*')))
        print(f'  {kind}/{split}: {n} files')

print('Done.')

# Write data.yaml pointing to local SSD copy
import yaml

data_yaml = {
    'path': str(DATA_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc': 1,
    'names': ['tree'],
}
yaml_path = DATA_DIR / 'data.yaml'
yaml_path.parent.mkdir(parents=True, exist_ok=True)
yaml_path.write_text(yaml.dump(data_yaml))
print(yaml_path.read_text())

In [ ]:
# Resume detection: check Drive for last.pt from a previous interrupted session
import shutil
from pathlib import Path

LOCAL_LAST = RUN_DIR / 'weights' / 'last.pt'
DRIVE_LAST = DRIVE_DIR / 'weights' / 'last.pt'

if DRIVE_LAST.exists():
    print(f'[Resume] Found last.pt on Drive, copying to local...')
    LOCAL_LAST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_LAST, LOCAL_LAST)
    print(f'[Resume] last.pt → {LOCAL_LAST}')
    print('[Resume] Will train with resume=True in Cell 5')
    RESUME = True
else:
    print('[Fresh] No last.pt on Drive — starting from scratch')
    RESUME = False

assert isinstance(RESUME, bool)

In [ ]:
import torch
from ultralytics import YOLO

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB')

if RESUME:
    # resume=True restores all hyperparams from checkpoint — do NOT re-pass them
    model = YOLO(str(LOCAL_LAST))
    results = model.train(resume=True)
else:
    model = YOLO('yolo26s.pt')
    results = model.train(
        data=str(yaml_path),
        epochs=100,
        imgsz=1280,
        batch=-1,             # autobatch (L4 23 GB VRAM → expect ~24-32)
        freeze=10,
        optimizer='auto',
        lr0=1e-3,
        lrf=0.01,
        cos_lr=True,
        patience=20,
        save=True,
        project='/content/runs/detect',
        name='tree_yolo26s_halfres',
        mosaic=0.0,
        mixup=0.0,
        degrees=10.0,
        translate=0.15,
        scale=0.5,
        flipud=0.0,
        fliplr=0.5,
        device=0,
        workers=4,
        cache=True,
        rect=True,
        deterministic=False,
    )

assert isinstance(model, YOLO)

In [ ]:
# Save weights and results back to Drive
import shutil

DRIVE_WEIGHTS = DRIVE_DIR / 'weights'
DRIVE_WEIGHTS.mkdir(parents=True, exist_ok=True)

for fname in ('best.pt', 'last.pt'):
    src = RUN_DIR / 'weights' / fname
    if src.exists():
        shutil.copy2(src, DRIVE_WEIGHTS / fname)
        print(f'[Saved] {fname} → {DRIVE_WEIGHTS}')
    else:
        print(f'[Warning] {fname} not found at {src}')

results_src = RUN_DIR / 'results.csv'
if results_src.exists():
    shutil.copy2(results_src, DRIVE_WEIGHTS / 'results.csv')
    print(f'[Saved] results.csv → {DRIVE_WEIGHTS}')

assert LOCAL_LAST.exists(), 'last.pt missing — check if training completed at least 1 epoch'
print('All weights saved to Drive.')

## Results & Next Steps

After training completes:
1. Check `results.csv` for mAP@0.5 trend — expect improvement over 26l baseline (0.811)
2. If mAP > 0.87 (26s Phase C baseline), the half-resolution strategy is validated
3. Download `best.pt` from Drive → `runs/detect/tree_yolo26s_halfres/weights/`
4. Run local inference on test frames to compare with GDINO quality

**Resume instructions** (if session interrupted):
- Re-run Cell 6 from the previous session first (saves `last.pt` to Drive if not done)
- Then re-run all cells — Cell 4 will detect `last.pt` on Drive and set `RESUME=True`
- Cell 5 will call `model.train(resume=True)` automatically

**Known risks**:
- NaN loss after resume (Ultralytics #3299/#17282): delete `last.pt` from Drive and retrain from scratch, or fine-tune from `best.pt`
- autobatch failure (`batch=-1`): manually set `batch=16` in Cell 5